# IOAI — 2025 Stage 3 Translation Stylization (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, subprocess, zipfile, urllib.request
subprocess.run(['pip','install','-q','sentencepiece','sacremoses'])
if not os.path.exists('data/train_dataset.jsonl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-translation-stylization/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 번역 문체화 (Translation Stylization, 베이스라인)

폴란드 AI 올림피아드 II · 2025 · 결선. 영어→폴란드어 번역 모델(`gsarti/opus-mt-tc-en-pl`)을, AI/ML **전문
용어(keywords)는 영어 그대로 남기는** 특정 문체로 맞춘다. (예: "explainable AI", "model predictions" 를
폴란드어로 번역하지 않고 그대로 둠 — 참조 번역이 그런 스타일이다.)

**채점**: 생성한 번역과 참조 번역의 **BLEU**(원문제 그대로 `sentence_bleu([ref], hyp)` — 문자열이라 **문자단위**
BLEU) 평균 → `compute_score`(≤0.82→0, 0.82~0.86 선형, ≥0.86→100).

이 노트북은 **베이스라인** = 사전학습 모델 그대로(용어까지 전부 번역) → BLEU 0.73 → **0점**. 모범답안(미세조정)을 참고하라.

**제출**: `submission.json` — valid 856문장의 생성 번역 리스트(순서 유지). *토크나이저는 변경 금지.*


In [ ]:
# 데이터 준비 + 모델 로드
import os, json, urllib.request, zipfile
if not os.path.exists("data/train_dataset.jsonl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-translation-stylization/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
dev = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "gsarti/opus-mt-tc-en-pl"

def load_jsonl(p):
    out = []
    for line in open(p, encoding="utf-8"):
        it = json.loads(line)
        out.append({"en": it["translation"]["en"], "pl": it["translation"]["pl"],
                    "keywords": ",".join(it.get("keywords", []))})
    return out
train = load_jsonl("data/train_dataset.jsonl"); valid = load_jsonl("data/valid_dataset.jsonl")

tokenizer = AutoTokenizer.from_pretrained(MODEL)   # 토크나이저 변경 금지
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).float().to(dev)   # fp32 (일부 버전은 fp16 로 로드→학습 NaN)
model.lm_head.weight = model.model.shared.weight   # 출력투영을 임베딩과 tie (일부 transformers 버전 로딩 보정)
print("train", len(train), "valid", len(valid), "| dev", dev)

def process_example(en: str, keywords: str) -> str:
    return en                                       # 항등 입력 (미세조정이 문체를 담당)

def generate_all(m, examples, bs=64):
    m.eval(); hyps = []
    for i in range(0, len(examples), bs):
        b = examples[i:i+bs]
        inp = [process_example(e["en"], e["keywords"]) for e in b]
        enc = tokenizer(inp, return_tensors="pt", padding=True, truncation=True, max_length=512).to(dev)
        with torch.no_grad(): out = m.generate(**enc, max_new_tokens=64, num_beams=4)
        hyps += tokenizer.batch_decode(out, skip_special_tokens=True)
    return hyps


In [ ]:
# 베이스라인: 미세조정 없이 사전학습 모델 그대로 사용
my_model = model


In [ ]:
# valid 생성 -> submission.json
hyps = generate_all(my_model, valid)
assert len(hyps) == len(valid)
json.dump(hyps, open("submission.json", "w"), ensure_ascii=False)
print("submission.json 저장:", len(hyps), "문장")
print("예시:", hyps[0][:90])


### 다음 단계
사전학습 모델은 용어까지 전부 폴란드어로 번역해 참조 문체와 어긋난다(BLEU 0.73 → 0점). train 쌍으로
**미세조정**하면 "용어는 영어로 유지" 문체를 학습해 BLEU 0.87 → 100점. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)